In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install requests pandas tqdm -q

import requests
import pandas as pd
from tqdm import tqdm
import time

API_KEY = "f2d68b86-4c0b-4fe7-87ef-c792cd510f29"
BASE_URL = "https://api.company-information.service.gov.uk"

test = requests.get(
    f"{BASE_URL}/advanced-search/companies",
    params={"sic_codes": "62020", "company_status": "active", "size": 1},
    auth=(API_KEY, "")
)
print(f"Status: {test.status_code}")
print(f"Sample: {test.json()}")

Status: 200
Sample: {'etag': 'eec675f925632431ab8f58e7a6b928dfe76edde1', 'top_hit': {'company_name': 'H J WHEELER LIMITED', 'company_number': '07394977', 'company_status': 'active', 'company_type': 'ltd', 'kind': 'search-results#company', 'links': {'company_profile': '/company/07394977'}, 'date_of_creation': '2010-10-04', 'registered_office_address': {'address_line_1': '36 Queens Road', 'locality': 'Newbury', 'postal_code': 'RG14 7NE', 'region': 'Berkshire'}, 'sic_codes': ['62020']}, 'items': [{'company_name': 'H J WHEELER LIMITED', 'company_number': '07394977', 'company_status': 'active', 'company_type': 'ltd', 'kind': 'search-results#company', 'links': {'company_profile': '/company/07394977'}, 'date_of_creation': '2010-10-04', 'registered_office_address': {'address_line_1': '36 Queens Road', 'locality': 'Newbury', 'postal_code': 'RG14 7NE', 'region': 'Berkshire'}, 'sic_codes': ['62020']}], 'kind': 'search#advanced-search', 'hits': 172499}


In [ ]:
SIC_CODES = [
    "62020",
    "62012",
    "62090",
    "63120",
    "62011",
    "63990",
    "66190",
    "64999",
]

def fetch_companies_by_sic(sic_code, max_results=10000):
    companies = []
    start_index = 0
    page_size = 100

    with tqdm(total=max_results, desc=f"SIC {sic_code}") as pbar:
        while len(companies) < max_results:
            response = requests.get(
                f"{BASE_URL}/advanced-search/companies",
                params={
                    "sic_codes": sic_code,
                    "company_status": "active",
                    "size": page_size,
                    "start_index": start_index
                },
                auth=(API_KEY, "")
            )

            if response.status_code == 429:
                print("Rate limited — waiting 10 seconds...")
                time.sleep(10)
                continue

            if response.status_code != 200:
                print(f"Error {response.status_code} for SIC {sic_code}")
                break

            data = response.json()
            items = data.get("items", [])
            if not items:
                print(f"  No more results at index {start_index}")
                break

            companies.extend(items)
            pbar.update(len(items))
            start_index += page_size
            time.sleep(0.5)

            if len(items) < page_size:
                print(f"  Reached end at {len(companies)} companies")
                break

    return companies[:max_results]

all_companies = []
for sic in SIC_CODES:
    results = fetch_companies_by_sic(sic, max_results=10000)
    print(f"SIC {sic}: {len(results)} companies fetched")
    all_companies.extend(results)

print(f"\nTotal before dedup: {len(all_companies)}")

SIC 62020: 100%|██████████| 10000/10000 [01:50<00:00, 90.24it/s]


SIC 62020: 10000 companies fetched


SIC 62012: 100%|██████████| 10000/10000 [01:52<00:00, 89.10it/s]


SIC 62012: 10000 companies fetched


SIC 62090: 100%|██████████| 10000/10000 [01:51<00:00, 89.55it/s]


...

SIC 63990: 10000 companies fetched


SIC 66190: 100%|██████████| 10000/10000 [01:52<00:00, 88.62it/s]


SIC 66190: 10000 companies fetched


SIC 64999: 100%|██████████| 10000/10000 [01:52<00:00, 88.59it/s]

SIC 64999: 10000 companies fetched

Total before dedup: 80000


In [ ]:
def parse_company(c):
    address = c.get("registered_office_address", {})
    sic_codes = c.get("sic_codes", [])
    return {
        "CompanyName": c.get("company_name", ""),
        "CompanyNumber": c.get("company_number", ""),
        "CompanyStatus": c.get("company_status", ""),
        "RegAddress.PostTown": address.get("locality", ""),
        "RegAddress.PostCode": address.get("postal_code", ""),
        "RegAddress.Country": address.get("country", ""),
        "SICCode.SicText_1": sic_codes[0] if sic_codes else "",
        "IncorporationDate": c.get("date_of_creation", ""),
    }

df = pd.DataFrame([parse_company(c) for c in all_companies])

df = df.drop_duplicates(subset="CompanyNumber").reset_index(drop=True)
print(f"Unique companies: {len(df)}")
print(df.head())

save_path = '/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_api.csv'
df.to_csv(save_path, index=False)
print("Saved companies_api.csv")

Unique companies: 74446
                             CompanyName CompanyNumber CompanyStatus  \
0                    H J WHEELER LIMITED      07394977        active   
1  RICHWIND SOFTWARE DEVELOPMENT LIMITED      07438022        active   
2              KENILWORTH CONSULTING LTD      07256308        active   
3                   TOLLINGTON DEYES LTD      06799271        active   
4                      MICHAEL W LIMITED      09982337        active   

  RegAddress.PostTown RegAddress.PostCode RegAddress.Country  \
0             Newbury            RG14 7NE                      
1              London             E15 4DJ                      
2            Loughton            IG10 3AG                      
3          Winchester            SO23 7RD                      
4          Manchester              M3 3HF            England   

  SICCode.SicText_1 IncorporationDate  
0             62020        2010-10-04  
1             62020        2010-11-12  
2             62020        2010-05-18 

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_api.csv')

print(f"Total rows: {len(df)}")
print(f"\nColumn dtypes:")
print(df.dtypes)
print(f"\nFirst SIC value: '{df['SICCode.SicText_1'].iloc[0]}' type: {type(df['SICCode.SicText_1'].iloc[0])}")
print(f"\nUnique SIC values (first 10):")
print(df['SICCode.SicText_1'].unique()[:10])
print(f"\nIncorporationDate sample:")
print(df['IncorporationDate'].head())

Total rows: 74446

Column dtypes:
CompanyName            object
CompanyNumber          object
CompanyStatus          object
RegAddress.PostTown    object
RegAddress.PostCode    object
RegAddress.Country     object
SICCode.SicText_1       int64
IncorporationDate      object
dtype: object

First SIC value: '62020' type: <class 'numpy.int64'>

Unique SIC values (first 10):
[62020 62012 62011 61900 49420 46190 61100 46510 41100 47910]

IncorporationDate sample:
0    2010-10-04
1    2010-11-12
2    2010-05-18
3    2009-01-22
4    2016-02-02
Name: IncorporationDate, dtype: object


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_api.csv')

target_sics = [62020, 62012, 62090, 63120, 62011, 63990, 66190, 64999]
df = df[df['SICCode.SicText_1'].isin(target_sics)].reset_index(drop=True)
print(f"After SIC filter: {len(df)}")

df['CompanyName'] = df['CompanyName'].str.upper()
df['RegAddress.PostTown'] = df['RegAddress.PostTown'].str.upper()

df['IncorporationDate'] = pd.to_datetime(df['IncorporationDate'], errors='coerce')
df = df[df['IncorporationDate'] >= '2010-01-01'].reset_index(drop=True)
print(f"After date filter: {len(df)}")

print(df['SICCode.SicText_1'].value_counts())

save_path = '/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_api_clean.csv'
df.to_csv(save_path, index=False)
print("Saved companies_api_clean.csv")

After SIC filter: 63048
After date filter: 52231
SICCode.SicText_1
62012    9479
62020    8475
64999    7077
62011    6260
66190    6105
63990    5669
62090    5443
63120    3723
Name: count, dtype: int64
Saved companies_api_clean.csv


In [ ]:
!pip install requests beautifulsoup4 tqdm -q

In [ ]:
import requests
import pandas as pd
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

df = pd.read_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_api_clean.csv')
print(f"Working with {len(df)} companies")

def clean_company_name(name):
    name = name.lower()
    removals = ['limited','ltd','plc','llp','llc','group','holdings',
                'consulting','consultancy','solutions','services',
                'technologies','technology','digital','software',
                'systems','uk','global']
    for word in removals:
        name = re.sub(r'\b' + word + r'\b', '', name)
    name = re.sub(r'[^a-z0-9\s]', '', name)
    name = re.sub(r'\s+', '', name).strip()
    return name

def guess_domains(company_name):
    clean = clean_company_name(company_name)
    if not clean or len(clean) < 3:
        return []
    return [
        f"https://www.{clean}.co.uk",
        f"https://www.{clean}.com",
        f"https://{clean}.io",
    ]

def find_website(row):
    for url in guess_domains(row['CompanyName']):
        try:
            r = requests.head(url, timeout=4, allow_redirects=True)
            if r.status_code < 400:
                return (row.name, url)
        except:
            continue
    return (row.name, None)

results = {}
with ThreadPoolExecutor(max_workers=50) as executor:
    futures = {executor.submit(find_website, row): row.name
               for _, row in df.iterrows()}
    for future in tqdm(as_completed(futures), total=len(df)):
        idx, url = future.result()
        results[idx] = url

df['website'] = df.index.map(results)
found = df['website'].notna().sum()
print(f"\nWebsites found: {found} / {len(df)} ({round(found/len(df)*100)}%)")

df.to_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_with_websites.csv', index=False)
print("Saved companies_with_websites.csv")

Working with 52231 companies


 65%|██████▍   | 33858/52231 [12:08<05:30, 55.53it/s]WARNING:urllib3.connection:Failed to parse headers (url=https://www.ukcraftfairs.com:443/): [MissingHeaderBodySeparatorDefect()], unparsed data: "Strict Transport Security: max-age=15552001; includeSubDomains; preload\r\nX-Frame-Options: SAMEORIGIN\r\nContent-Security-Policy: frame-ancestors 'none'; base-uri 'self';\r\nDate: Tue, 26 May 2026 16:35:09 GMT\r\n\r\n"
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: "Strict Transport Security: max-age=15552001; includeSubDomains; preload\r\nX-Frame-Options: SAMEORIGIN\r\nContent-Security-P


Websites found: 24659 / 52231 (47%)
Saved companies_with_websites.csv


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

df = pd.read_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_with_websites.csv')
df_with = df[df['website'].notna()].copy().reset_index(drop=True)
print(f"Verifying {len(df_with)} companies...")

def get_page_text(url, timeout=6):
    try:
        r = requests.get(url, timeout=timeout, headers={
            'User-Agent': 'Mozilla/5.0 (compatible; research-bot/1.0)'
        })
        soup = BeautifulSoup(r.text, 'html.parser')
        return soup.get_text(separator=' ', strip=True).lower()
    except:
        return ''

def verify_website(row):
    url = row['website']
    company = row['CompanyName'].lower()
    town = str(row['RegAddress.PostTown']).lower()
    postcode = str(row['RegAddress.PostCode']).lower().split()[0]

    clean = re.sub(r'\b(limited|ltd|plc|llp|consulting|solutions|services|technologies|technology|digital|software|systems|group)\b', '', company)
    clean = re.sub(r'[^a-z0-9\s]', '', clean).strip()
    keywords = [w for w in clean.split() if len(w) > 3]

    text = get_page_text(url)
    if not text:
        return (row.name, False)

    domain = url.replace('https://', '').replace('http://', '').replace('www.', '').split('.')[0]
    domain_match = any(kw in domain for kw in keywords)
    keyword_matches = sum(1 for kw in keywords if kw in text)
    keyword_match = keyword_matches >= max(1, len(keywords) // 2)
    location_match = town in text or postcode in text

    verified = domain_match and (keyword_match or location_match)
    return (row.name, verified)

results = {}
with ThreadPoolExecutor(max_workers=30) as executor:
    futures = {executor.submit(verify_website, row): row.name
               for _, row in df_with.iterrows()}
    for future in tqdm(as_completed(futures), total=len(df_with)):
        idx, verified = future.result()
        results[idx] = verified

df_with['verified'] = df_with.index.map(results)
verified_df = df_with[df_with['verified'] == True].reset_index(drop=True)

print(f"\nVerified: {len(verified_df)} / {len(df_with)}")
print(f"Rejected: {len(df_with) - len(verified_df)}")

verified_df.to_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_verified.csv', index=False)
print("Saved companies_verified.csv")

Verifying 24659 companies...


 48%|████▊     | 11834/24659 [08:36<08:45, 24.38it/s]/tmp/ipykernel_675/2957894097.py:17: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(r.text, 'html.parser')
 72%|███████▏  | 17690/24659 [12:37<16:12,  7.16it/s]WARNING:urllib3.connection:Failed to parse headers (url=https://www.ukcraftfairs.com:443


Verified: 14172 / 24659
Rejected: 10487
Saved companies_verified.csv


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import warnings
from bs4 import XMLParsedAsHTMLWarning
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

df = pd.read_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_verified.csv')
print(f"Scraping {len(df)} verified companies...")

def scrape_website(row):
    url = row['website']
    try:
        r = requests.get(url, timeout=8, headers={
            'User-Agent': 'Mozilla/5.0 (compatible; research-bot/1.0)'
        })
        soup = BeautifulSoup(r.text, 'html.parser')
        for tag in soup(['nav','footer','header','script','style','aside']):
            tag.decompose()
        title = soup.title.string.strip() if soup.title else ''
        meta = soup.find('meta', attrs={'name': 'description'})
        meta_desc = meta['content'].strip() if meta and meta.get('content') else ''
        body = soup.get_text(separator=' ', strip=True)
        body = ' '.join(body.split())[:2000]
        full_text = f"{title}. {meta_desc}. {body}"
        return (row.name, full_text)
    except:
        return (row.name, None)

results = {}
with ThreadPoolExecutor(max_workers=30) as executor:
    futures = {executor.submit(scrape_website, row): row.name
               for _, row in df.iterrows()}
    for future in tqdm(as_completed(futures), total=len(df)):
        idx, text = future.result()
        results[idx] = text

df['scraped_text'] = df.index.map(results)
before = len(df)
df = df[df['scraped_text'].notna()].reset_index(drop=True)
print(f"\nSuccessfully scraped: {len(df)} / {before}")

df.to_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_scraped.csv', index=False)
print("Saved companies_scraped.csv")

Scraping 14172 verified companies...


100%|██████████| 14172/14172 [14:50<00:00, 15.92it/s]



Successfully scraped: 14067 / 14172
Saved companies_scraped.csv


In [ ]:
import pandas as pd
import re
import warnings
from bs4 import XMLParsedAsHTMLWarning
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

df = pd.read_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_scraped.csv')

def clean_text(text):
    if not isinstance(text, str):
        return None
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    boilerplate = ['cookie','privacy policy','terms of service',
                   'all rights reserved','copyright','gdpr',
                   'we use cookies','accept cookies','sign in','log in']
    for phrase in boilerplate:
        text = text.replace(phrase, '')
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if len(text) > 100 else None

df['clean_text'] = df['scraped_text'].apply(clean_text)
before = len(df)
df = df[df['clean_text'].notna()].reset_index(drop=True)
print(f"Dropped: {before - len(df)}")
print(f"Clean dataset: {len(df)} companies")

df.to_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_clean.csv', index=False)
print("Saved companies_clean.csv")

Dropped: 890
Clean dataset: 13177 companies
Saved companies_clean.csv


In [ ]:
!pip install sentence-transformers -q

from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_clean.csv')
print(f"Embedding {len(df)} companies...")

model = SentenceTransformer('all-MiniLM-L6-v2')
texts = df['clean_text'].tolist()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"Embeddings shape: {embeddings.shape}")

np.save('/content/drive/MyDrive/Company_Intelligence_Extractor_api/embeddings.npy', embeddings)

df_meta = df.drop(columns=['scraped_text', 'clean_text'])
df_meta.to_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_meta.csv', index=False)

print("Saved embeddings.npy and companies_meta.csv")

Embedding 13177 companies...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/206 [00:00<?, ?it/s]

Embeddings shape: (13177, 384)
Saved embeddings.npy and companies_meta.csv


In [ ]:
!pip install hdbscan umap-learn plotly -q

import numpy as np
import pandas as pd
import hdbscan
import umap

embeddings = np.load('/content/drive/MyDrive/Company_Intelligence_Extractor_api/embeddings.npy')
df = pd.read_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_meta.csv')

print("UMAP 10D for clustering...")
reducer_10d = umap.UMAP(
    n_components=10,
    n_neighbors=15,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)
embedding_10d = reducer_10d.fit_transform(embeddings)

print("Clustering with HDBSCAN...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=20,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom'
)
labels = clusterer.fit_predict(embedding_10d)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise = (labels == -1).sum()
print(f"\nClusters: {n_clusters}")
print(f"Noise: {noise}")
print(f"Clustered: {len(df) - noise}")
print(pd.Series(labels).value_counts().sort_index())

print("\nUMAP 2D for visualisation...")
reducer_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
embedding_2d = reducer_2d.fit_transform(embeddings)

df['x'] = embedding_2d[:, 0]
df['y'] = embedding_2d[:, 1]
df['cluster'] = labels

df.to_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_clustered.csv', index=False)
print("Saved companies_clustered.csv")

UMAP 10D for clustering...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Clustering with HDBSCAN...

Clusters: 87
Noise: 4082
Clustered: 9095
-1     4082
 0       35
 1       26
 2       25
 3      365
       ... 
 82      49
 83      42
 84      26
 85     185
 86     278
Name: count, Length: 88, dtype: int64

UMAP 2D for visualisation...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved companies_clustered.csv


In [ ]:
import numpy as np
import pandas as pd
import hdbscan
import umap

embeddings = np.load('/content/drive/MyDrive/Company_Intelligence_Extractor_api/embeddings.npy')
df = pd.read_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_meta.csv')

print("UMAP 10D...")
reducer_10d = umap.UMAP(
    n_components=10, n_neighbors=15,
    min_dist=0.0, metric='cosine', random_state=42
)
embedding_10d = reducer_10d.fit_transform(embeddings)

print("Clustering...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=150,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom'
)
labels = clusterer.fit_predict(embedding_10d)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise = (labels == -1).sum()
print(f"Clusters: {n_clusters}")
print(f"Noise: {noise} ({round(noise/len(df)*100)}%)")
print(pd.Series(labels).value_counts().sort_index())

UMAP 10D...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Clustering...
Clusters: 18
Noise: 2372 (18%)
-1     2372
 0      176
 1      378
 2      365
 3      188
 4      168
 5      592
 6      178
 7     5578
 8      508
 9      364
 10     405
 11     414
 12     542
 13     190
 14     237
 15     165
 16     181
 17     176
Name: count, dtype: int64


In [ ]:
import umap
import plotly.express as px
import numpy as np
import pandas as pd

embeddings = np.load('/content/drive/MyDrive/Company_Intelligence_Extractor_api/embeddings.npy')
df = pd.read_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_meta.csv')

print("UMAP 2D for visualisation...")
reducer_2d = umap.UMAP(
    n_components=2, n_neighbors=15,
    min_dist=0.1, metric='cosine', random_state=42
)
embedding_2d = reducer_2d.fit_transform(embeddings)

df['x'] = embedding_2d[:, 0]
df['y'] = embedding_2d[:, 1]
df['cluster'] = labels
df['cluster_label'] = df['cluster'].apply(
    lambda x: f"Cluster {x}" if x != -1 else "Unclustered"
)

df.to_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_clustered.csv', index=False)

fig = px.scatter(
    df[df['cluster'] != -1],
    x='x', y='y',
    color='cluster_label',
    hover_data={
        'CompanyName': True,
        'RegAddress.PostTown': True,
        'SICCode.SicText_1': True,
        'website': True,
        'x': False, 'y': False,
        'cluster_label': False
    },
    title='UK Tech and Fintech Companies — Clustered by Website Content',
    width=1200, height=800
)
fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_layout(legend_title_text='Cluster')

fig.write_html('/content/drive/MyDrive/Company_Intelligence_Extractor_api/clusters_final.html')
print("Saved clusters_final.html")
print(f"\nFinal summary:")
print(f"Total companies: {len(df)}")
print(f"Clusters: 18")
print(f"Clustered: {(df['cluster'] != -1).sum()}")
print(f"Noise: {(df['cluster'] == -1).sum()}")

UMAP 2D for visualisation...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved clusters_final.html

Final summary:
Total companies: 13177
Clusters: 18
Clustered: 10805
Noise: 2372


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_clustered.csv')
for i in [5, 7, 8, 12]:
    print(f"\nCluster {i} ({len(df[df['cluster']==i])}):")
    print(df[df['cluster']==i][['CompanyName','website']].head(5).to_string())


Cluster 5 (592):
                            CompanyName                          website
12                 SPRING INFO-TECH LTD   https://www.springinfotech.com
64                   CHANGE HARBOUR LTD  https://www.changeharbour.co.uk
91               SMART TECH SOL LIMITED   https://www.smarttechsol.co.uk
124                    STRATEGY LED LTD      https://www.strategyled.com
145  LAKSH CONSULTANCY SERVICES LIMITED          https://www.laksh.co.uk

Cluster 7 (5578):
            CompanyName                       website
1     FUNCTION1 LIMITED     https://www.function1.com
2  UNREAL SOLUTIONS LTD             https://unreal.io
3          DOX 4DEV LTD     https://www.dox4dev.co.uk
4         BITSECURE LTD     https://www.bitsecure.com
5    MOTIONLOVE LIMITED  https://www.motionlove.co.uk

Cluster 8 (508):
               CompanyName                       website
187         PREDICTIVA LTD  https://www.predictiva.co.uk
223  ALURE TRADING LIMITED  https://www.aluretrading.com
308    SDDC 

In [ ]:
for i in [9, 13, 14, 16]:
    print(f"\nCluster {i} ({len(df[df['cluster']==i])}):")
    print(df[df['cluster']==i][['CompanyName','website']].head(5).to_string())


Cluster 9 (364):
                        CompanyName                           website
487         X CAPITAL GROUP LIMITED          https://www.xcapital.com
1710  LOTI SOFTWARE CONSULTANCY LTD              https://www.loti.com
2669               DYNAMO GROUP LTD          https://www.dynamo.co.uk
2779  SCHOENBERG TECHNOLOGY LIMITED        https://www.schoenberg.com
2949        MORTGAGE WALLET LIMITED  https://www.mortgagewallet.co.uk

Cluster 13 (190):
                        CompanyName                     website
200        SHORELINE TECHNOLOGY LTD   https://www.shoreline.com
503  BLUSKY TECHNOLOGY SERVICES LTD      https://www.blusky.com
547                 SUBVERSIVE LTD.  https://www.subversive.com
587           AMARTYA GROUP LIMITED   https://www.amartya.co.uk
636                     ASTORTS LTD     https://www.astorts.com

Cluster 14 (237):
                               CompanyName                                       website
683            MO FINANCIAL CONSULTING LTD         

In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_clustered.csv')

labels = {
    -1: 'Unclustered',
     0: 'Web and digital services',
     1: 'Generic IT consultancy',
     2: 'Cybersecurity and data',
     3: 'Data and strategy consulting',
     4: 'Software development studios',
     5: 'International IT services',
     6: 'IT training and education',
     7: 'General IT and software',
     8: 'Business consulting and trading',
     9: 'Financial services and fintech',
    10: 'Health tech and wellness',
    11: 'UK regional consultancies',
    12: 'Professional services',
    13: 'IT outsourcing and managed services',
    14: 'Financial software and lending',
    15: 'Strategy and research',
    16: 'Global business services',
    17: 'Creative and lifestyle tech',
}

df['cluster_label'] = df['cluster'].map(labels)

fig = px.scatter(
    df[df['cluster'] != -1],
    x='x', y='y',
    color='cluster_label',
    hover_name='cluster_label',
    hover_data={
        'CompanyName': True,
        'RegAddress.PostTown': True,
        'SICCode.SicText_1': True,
        'website': True,
        'x': False, 'y': False,
        'cluster_label': False
    },
    title='UK Tech and Fintech Companies — Clustered by Website Content',
    width=1200, height=800
)
fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_layout(legend_title_text='Cluster')

df.to_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_final.csv', index=False)
fig.write_html('/content/drive/MyDrive/Company_Intelligence_Extractor_api/clusters_final_labelled.html')
print("Saved")

Saved


In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('/content/drive/MyDrive/Company_Intelligence_Extractor_api/companies_final.csv')

df_plot = df[df['cluster'] != -1].copy()

fig = px.scatter(
    df_plot,
    x='x', y='y',
    color='cluster_label',
    hover_name='cluster_label',
    hover_data={
        'CompanyName': True,
        'RegAddress.PostTown': True,
        'website': True,
        'SICCode.SicText_1': False,
        'x': False,
        'y': False,
        'cluster_label': False
    },
    title='UK Tech and Fintech Companies — Clustered by Website Content',
    width=1200,
    height=800
)

fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_layout(
    legend_title_text='Cluster',
    legend=dict(
        itemsizing='constant',
        font=dict(size=12)
    ),
    hoverlabel=dict(
        bgcolor='white',
        font_size=13
    )
)

fig.write_html('/content/drive/MyDrive/Company_Intelligence_Extractor_api/clusters_final_labelled.html')
print("Done")

Done


##Phase 2 — Companies House API enrichment
   (officer names, filing activity, accounts)
- Re-cluster with enriched features
- Compare Phase 1 vs Phase 2 clusters
